In [6]:
import pandas as pd
import numpy as np
from sklearn.cluster import OPTICS
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neighbors import NearestNeighbors

# seleccion data frame
df = pd.read_csv(r"C:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\Datos\TrainX.csv")


#### No hace falta limpiar los datos, transformar, ni normalizar puesto que este proceso se hizo con el dataset original

#### No es necesario conseguir coeficiente de silueta/codo para OPTICS

# a. Tomar minimo 3 hiperparametros

In [18]:
nbrs = NearestNeighbors(n_neighbors=5).fit(df)
distances, _ = nbrs.kneighbors(df)
print("Promedio de distancia al 5° vecino:", np.mean(distances[:, -1]))

Promedio de distancia al 5° vecino: 2.9402513395792194


In [ ]:
# numero minimo de punto para formar un cluster (minPts)
min_samples_values = [5, 8, 10]
# maximo radio de vecinidad (ε)
max_eps_values = [4]
# umbral para determinar la separacion de clusters
xi_values = [0.03, 0.05, 0.1]

# b. Sacar grupos, con parametros diferentes
#### Cambiando 3 veces el valor de cada hiperparámetro y aplicar el algoritmo varias veces (varios for's anidados para cambiar los hiperparámetros)

In [ ]:
# ===============================================================
# OPTICS: búsqueda de hiperparámetros coherentes
# ===============================================================

from sklearn.cluster import OPTICS
from sklearn.metrics import silhouette_score
import numpy as np
import pandas as pd

# === Hiperparámetros ajustados según tu dataset ===
min_samples_values = [3, 5, 10]
max_eps_values = [2.0, 3.0, 4.0]
xi_values = [0.03, 0.05, 0.1]

# === DataFrame para almacenar resultados ===
resultados = []

# === Búsqueda en rejilla (3 for anidados) ===
for min_samples in min_samples_values:
    for max_eps in max_eps_values:
        for xi in xi_values:
            try:
                optics = OPTICS(min_samples=min_samples, max_eps=max_eps, xi=xi)
                optics.fit(df)
                labels = optics.labels_

                # Puntos que no son ruido
                mask = labels != -1
                num_clusters = len(set(labels)) - (1 if -1 in labels else 0)
                noise_pct = np.mean(labels == -1) * 100

                # Calcular silueta solo si hay más de 1 clúster válido
                if num_clusters > 1 and np.sum(mask) > 1:
                    sil = silhouette_score(df[mask], labels[mask])
                else:
                    sil = np.nan

                resultados.append({
                    'min_samples': min_samples,
                    'max_eps': max_eps,
                    'xi': xi,
                    'n_clusters': num_clusters,
                    'ruido_%': round(noise_pct, 2),
                    'silhouette': round(sil, 4) if not np.isnan(sil) else None
                })
            except Exception as e:
                resultados.append({
                    'min_samples': min_samples,
                    'max_eps': max_eps,
                    'xi': xi,
                    'n_clusters': None,
                    'ruido_%': None,
                    'silhouette': None,
                    'error': str(e)
                })

# === Convertir resultados a DataFrame ===
tabla_resultados = pd.DataFrame(resultados)

# === Ordenar por coeficiente de silueta (descendente) ===
tabla_resultados = tabla_resultados.sort_values(
    by='silhouette', ascending=False, na_position='last'
).reset_index(drop=True)

# === Mostrar tabla resumen ===
print("Resultados comparativos de OPTICS:")
display(tabla_resultados)

# === Mostrar mejor configuración encontrada ===
mejor_fila = tabla_resultados.iloc[0]
print("\n🔹 Mejor configuración encontrada:")
print(mejor_fila)


Resultados comparativos de OPTICS:


,min_samples,max_eps,xi,n_clusters,ruido_%,silhouette
0,6,4.0,0.03,3,95.31,0.3638
1,6,8.0,0.03,3,95.31,0.3638
2,6,12.0,0.03,3,95.31,0.3638
3,7,4.0,0.03,2,96.68,0.2920
4,7,4.0,0.05,2,96.68,0.2920
5,7,8.0,0.03,2,96.68,0.2920
6,7,8.0,0.05,2,96.68,0.2920
7,7,12.0,0.03,2,96.68,0.2920
8,7,12.0,0.05,2,96.68,0.2920
9,5,4.0,0.03,6,92.77,0.2483



🔹 Mejor configuración encontrada:
min_samples     6.0000
max_eps         4.0000
xi              0.0300
n_clusters      3.0000
ruido_%        95.3100
silhouette      0.3638
Name: 0, dtype: float64


In [26]:
for min_s in min_samples_values:
    for eps in max_eps_values:
        for xi in xi_values:
            optics = OPTICS(min_samples=min_s, max_eps=eps, xi=xi)
            df['cluster'] = optics.fit_predict(df)
            print(df["cluster"].value_counts())

cluster
-1    489
 2      5
 0      3
 1      3
 3      3
 4      3
 5      3
 6      3
Name: count, dtype: int64
cluster
-1    487
 3      6
 2      4
 0      3
 1      3
 4      3
 5      3
 6      3
Name: count, dtype: int64
cluster
-1    491
 2      6
 0      3
 1      3
 3      3
 4      3
 5      3
Name: count, dtype: int64
cluster
-1     353
 5      11
 2       9
 21      8
 29      7
 17      7
 10      6
 31      6
 1       6
 4       5
 30      5
 20      5
 7       5
 22      5
 26      5
 18      4
 8       4
 11      4
 12      4
 3       4
 0       4
 19      3
 34      3
 33      3
 13      3
 32      3
 6       3
 28      3
 15      3
 9       3
 25      3
 16      3
 23      3
 27      3
 14      3
 24      3
Name: count, dtype: int64
cluster
-1     329
 15     10
 3       9
 19      8
 22      7
 17      7
 2       6
 38      6
 26      6
 33      5
 32      5
 24      5
 16      5
 29      4
 9       4
 27      4
 10      4
 39      4
 35      4
 20      4
 7       4

c:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\proyectoIAEnv\Lib\site-packages\sklearn\cluster\_optics.py:664: UserWarning: All reachability values are inf. Set a larger max_eps or all data will be considered outliers.
  warnings.warn(


cluster
-1    470
 3     10
 1      7
 0      5
 5      5
 6      5
 4      5
 2      5
Name: count, dtype: int64
cluster
-1    473
 1      7
 6      7
 0      5
 3      5
 4      5
 5      5
 2      5
Name: count, dtype: int64
cluster
-1    485
 1      7
 0      5
 3      5
 4      5
 2      5
Name: count, dtype: int64
cluster
-1    458
 3      7
 6      7
 5      5
 8      5
 2      5
 9      5
 4      5
 1      5
 7      5
 0      5
Name: count, dtype: int64
cluster
-1    463
 1      7
 3      7
 2      5
 5      5
 6      5
 8      5
 7      5
 4      5
 0      5
Name: count, dtype: int64
cluster
-1    475
 2      7
 1      5
 4      5
 5      5
 6      5
 3      5
 0      5
Name: count, dtype: int64
cluster
-1    512
Name: count, dtype: int64
cluster
-1    512
Name: count, dtype: int64
cluster
-1    512
Name: count, dtype: int64


c:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\proyectoIAEnv\Lib\site-packages\sklearn\cluster\_optics.py:664: UserWarning: All reachability values are inf. Set a larger max_eps or all data will be considered outliers.
  warnings.warn(
c:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\proyectoIAEnv\Lib\site-packages\sklearn\cluster\_optics.py:664: UserWarning: All reachability values are inf. Set a larger max_eps or all data will be considered outliers.
  warnings.warn(
c:\Users\mejia\Desktop\Estudio\IA\Proyecto4-IA\proyectoIAEnv\Lib\site-packages\sklearn\cluster\_optics.py:664: UserWarning: All reachability values are inf. Set a larger max_eps or all data will be considered outliers.
  warnings.warn(


cluster
-1    504
 0      8
Name: count, dtype: int64
cluster
 0    395
-1    117
Name: count, dtype: int64
cluster
 0    395
-1    117
Name: count, dtype: int64
cluster
-1    504
 0      8
Name: count, dtype: int64
cluster
 0    511
-1      1
Name: count, dtype: int64
cluster
 0    511
-1      1
Name: count, dtype: int64


In [27]:
optics = OPTICS(min_samples=5, max_eps=3.0, xi=0.05)
optics.fit(df)
labels = optics.labels_

print("Clusters encontrados:", len(set(labels)) - (1 if -1 in labels else 0))
print("Porcentaje de ruido:", np.mean(labels == -1))


Clusters encontrados: 1
Porcentaje de ruido: 0.986328125


# c. 